# 11: `aftermarket audit` proof run on GC (Kaggle or Colab)
End-to-end check of the audit tool: run `aftermarket audit --quick` on a model whose numbers we already have, and see
whether it reproduces them. The model is GC (Guardián colombiano, `vasmi-sas/guardian-tiny-aya-lisa`, a LoRA adapter on
Tiny Aya Global), read from `configs/models.yaml` at its pinned revision, under the audit name `audit-gc`.

The quick battery (A0, A2, A3, A5bon on the fixed 100-prompt subsets) is generated and judged from scratch by the
audit command, exactly as a user would run it: nothing is reused from the leaderboard runs. The card's retention is
then compared with GC's leaderboard retention on the same 100 prompts (`results/tables/quick_retention.csv`:
0.57 [0.52–0.62]). **Reproduced** = the audit's retention lies inside that 95% CI (`validation.json`; the
per-attack cells are compared the same way, for information).

The card, its aggregate tables, the manifest and `validation.json` are pushed to the PRIVATE dataset
`<you>/aftermarket-runs` under `audit/audit-gc/`. Generations and judged rows stay in the session and are never pushed.

- **Kaggle**: GPU T4 x2, secret `HF_TOKEN` (Add-ons -> Secrets; Tiny Aya, Llama Guard 3 and WildGuard licences
  accepted), then **Save & Run All (Commit)**. Estimate: ~1-1.5 h.
- **Colab**: Runtime -> T4 GPU, secret `HF_TOKEN`, Run all.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bennytimz/aftermarket/blob/main/notebooks/11_audit_demo.ipynb)

In [ ]:
# Session clock, platform. Kaggle charges the whole session (setup included), so the clock starts here.
import os, sys, time
SESSION_START = time.time()
IN_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.isdir("/kaggle") and not IN_COLAB
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if IN_COLAB else "other")

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import subprocess
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping the image's torch build fixed.
#     Colab: A1 trains a QLoRA adapter, so bitsandbytes / peft / accelerate are pinned as in requirements.txt.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
if IN_COLAB:
    !pip install -q bitsandbytes==0.50.2 peft==0.21.2 accelerate==1.15.0 -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face: Colab secrets (google.colab.userdata) on Colab, Kaggle secrets otherwise.
# Never prompts, so it also works in background "Save & Run All" runs.
from huggingface_hub import login, whoami
HF_USER = None
try:
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")  # key icon -> HF_TOKEN, notebook access on
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")  # Add-ons -> Secrets
    login(token=token, add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info: every visible T4 gets one worker (Kaggle: 2, Colab: 1)
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
GPUS = ",".join(str(i) for i in range(max(torch.cuda.device_count(), 1)))
print("GPUS =", GPUS)

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import os, json, sys
os.environ["AFTERMARKET_DEFENSE_DIR"] = "/kaggle/temp/aftermarket_defense" if os.path.isdir("/kaggle") else "/tmp/aftermarket_defense"
os.environ["AFTERMARKET_ADAPTERS"] = "/kaggle/temp/aftermarket_adapters" if os.path.isdir("/kaggle") else "/tmp/aftermarket_adapters"
os.environ["AFTERMARKET_TMP"] = "/kaggle/temp" if os.path.isdir("/kaggle") else "/tmp"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
for k in ("AFTERMARKET_DEFENSE_DIR", "AFTERMARKET_ADAPTERS"):
    os.makedirs(os.environ[k], exist_ok=True)
sys.path.insert(0, "src")
print("gpus:", GPUS)

In [ ]:
# Install the `aftermarket` command itself (dependencies are already in place from the cell above)
!pip install -q -e . --no-deps
!aftermarket audit --help | head -3

## 1. The model under audit: GC, exactly as pinned in configs/models.yaml

In [ ]:
import yaml
reg = yaml.safe_load(open("configs/models.yaml", encoding="utf-8"))
GC = reg["models"]["GC"]
assert GC["format"] == "lora" and GC.get("base") == "BASE"
ADAPTER, REVISION, BASE_ID = GC["hf_path"], GC["revision"], reg["base_model"]
print(f"base {BASE_ID} + adapter {ADAPTER} @ {REVISION}")

## 2. Run the audit (generate -> judge -> card), as a user would
One worker per GPU; generation and the judges run in separate processes. Re-running in the same session resumes.

In [ ]:
!aftermarket audit --model {BASE_ID} --adapter {ADAPTER} --revision {REVISION} --quick --name audit-gc --gpus {GPUS}; echo $? > /tmp/audit_exit
exit_code = int(open("/tmp/audit_exit").read().strip())
assert exit_code == 0, f"aftermarket audit failed (exit {exit_code}); see the log above"

## 3. The report card

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
OUT = Path("results/audit/audit-gc")
display(Image(filename=str(OUT / "card" / "audit-gc.png")))
card_md = (OUT / "card" / "audit-gc.md").read_text(encoding="utf-8")
display(Markdown(card_md[card_md.index("**"):]))  # from the summary line on (title + image shown above)

## 4. Does it reproduce GC's leaderboard numbers on the same 100 prompts?

In [ ]:
from aftermarket.audit import validate
v = validate(OUT, "GC", quick=True)
r = v["retention"]
print(f"retention: audit {r['audit']:.3f} [{r['audit_ci'][0]:.2f}-{r['audit_ci'][1]:.2f}] vs GC leaderboard "
      f"{r['reference']:.3f} [{r['reference_ci'][0]:.2f}-{r['reference_ci'][1]:.2f}] on {'the same' if r['families_match'] else 'DIFFERENT'} families")
for c in v["cells"]:
    print(f"  {c['attack']:6s} audit {c['audit']:.0%}  GC {c['reference']:.0%} "
          f"[{c['reference_ci'][0]:.0%}-{c['reference_ci'][1]:.0%}]  {'within CI' if c['within_ci'] else 'OUTSIDE CI'}")
print("REPRODUCED" if v["reproduced"] else "NOT REPRODUCED", "(audit retention inside GC's 95% CI)")

## 5. Push the card + aggregates (never generations) to the private Hub dataset

In [ ]:
from aftermarket.audit import push_card
if HF_USER:
    print("pushed to", push_card(OUT))
else:
    print("no HF login: card left in", OUT)